# Preprocessing all histogram plotting data

This was done in the plotting notebook in the submission version, but takes quite long on a small machine. We decided to refactor this and put the histogram output on the public bucket so folks can access it publicly. 

In [1]:
# !pip install ../.

In [2]:
from distributed import Client
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 8,Total memory: 13.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37257,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 8
Started: Just now,Total memory: 13.63 GiB
Comm: tcp://127.0.0.1:36823,Total threads: 2
Dashboard: http://127.0.0.1:43219/status,Memory: 3.41 GiB
Nanny: tcp://127.0.0.1:37037,


In [3]:
import functools
import numpy as np
import xarray as xr
from xhistogram.xarray import histogram
from scale_aware_air_sea.parameters import get_params

import fsspec
fs = fsspec.filesystem(
    's3',
    profile='osn-scale-aware',  ## The profile name that is configured in the aws credentials file.
    client_kwargs={'endpoint_url': 'https://nyu1.osn.mghpcc.org'} # This is the endpoint for the m2lines osn pod
)

params = get_params("v1.0.1", test=False)
models = ["CM26", "CESM"]
wbc_boxes = [(-85, -40, 25, 45), (-235, -190, 25, 45)]
bins = {
    "qh": np.arange(-1200, 300, 3),
    "ql": np.arange(-1200, 300, 3),
    "q_total": np.arange(-1200, 300, 3),
}
small_scale_bins = np.arange(-300, 200, 1)

def mask_boxes(ds:xr.Dataset, boxes: list[tuple[float, float, float, float]]) -> xr.DataArray:
    """Masks out a list of boxes provided as  (lon_min, lon_max, lat_min, lat_max) values"""
    lon = ds.geolon_t
    lat = ds.geolat_t
    # detect the data lon convention
    lon_min = lon.min().load()
    box_lon_min = min([b[0] for b in boxes])
    # adjust the lon convention for the boxes if values are out of bound
    # this does not currently cover all cases, so check carefully!
    if lon_min>box_lon_min:
        boxes = [(b[0]+360, b[1]+360, b[2], b[3]) for b in boxes]
    global_min_lon = [b[0] for b in boxes]
    # create a mask for each box
    masks = []
    for box in boxes:
        lon_min, lon_max, lat_min, lat_max = box
        mask_single = np.logical_and(np.logical_and(lat>lat_min, lat<lat_max), np.logical_and(lon>lon_min, lon<lon_max))
        masks.append(mask_single)
    mask_combined = functools.reduce(np.logical_or, masks)
    return mask_combined


hist_dict_relative = {}

for model in models:
    path = params["paths"][model]["results"]["filter"]["native"]["appendix"]
    ds_appendix = xr.open_dataset(fs.get_mapper(path), engine="zarr", chunks={})
    ds_appendix["q_total"] = ds_appendix["ql"] + ds_appendix["qh"]
    
    hist_dict_relative[model] = {}
    for var in ["qh", "ql", "q_total"]:
        print(f"Processing Histogram for {model} {var}")
        da_plot = ds_appendix[var]

        highres = da_plot.sel(term="Q_H_bar")
        highres.name = f"High Resolution {var}"

        small_scale = da_plot.sel(term="Q_star_star")
        small_scale.name = f"Small Scale {var}"

        large_scale = highres - small_scale
        large_scale.name = f"Large Scale {var}"

        wbc_mask = mask_boxes(da_plot, wbc_boxes)
        wbc_mask.name = 'WBC_mask'

        # h_rel = histogram(highres, small_scale, bins=[bins[var],  small_scale_bins], dim=['xt_ocean', 'yt_ocean'])
        h = histogram(
            large_scale,
            small_scale,
            wbc_mask, # this splits the results into an additional dimesion (all points outside of wbc boxes, and all within)
            bins=[bins[var], small_scale_bins, np.array([-1,0.5,2])],
            dim=["xt_ocean", "yt_ocean"],
        )
        # avoid that stupid divide by 0 error
        h = h.sum("time")
        hist_dict_relative[model][var] = h.load()

Processing Histogram for CM26 qh


/srv/conda/envs/notebook/lib/python3.10/site-packages/dask/array/reductions.py:611: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)


Processing Histogram for CM26 ql


/srv/conda/envs/notebook/lib/python3.10/site-packages/dask/array/reductions.py:611: RuntimeWarning: All-NaN slice encountered
  return np.nanmin(x_chunk, axis=axis, keepdims=keepdims)


Processing Histogram for CM26 q_total


2025-05-01 00:46:42,254 - distributed.worker.memory - WARNING - gc.collect() took 3.610s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2025-05-01 00:46:46,507 - distributed.worker.memory - WARNING - gc.collect() took 3.482s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2025-05-01 00:46:58,338 - distributed.worker.memory - WARNING - gc.collect() took 2.723s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2025-05-01 00:48:52,589 - distributed.worker.memory - WARNING - gc.collect() took 5.033s. This is usually a sign that some tasks handle too many Python objects at the same time. Rechunking the work into smaller tasks might help.
2025-05-01 00:49:10,215 - distributed.worker.memory - WARNING - gc.collect() took 4.

Processing Histogram for CESM qh
Processing Histogram for CESM ql
Processing Histogram for CESM q_total


In [48]:
for model in models:
    datasets = []
    for variable in hist_dict_relative[model].keys():
        da = hist_dict_relative[model][variable]
        ds = da.to_dataset(name=f'histogram_{variable}')
        path = params['paths'][model]['plotting']['histogram'][variable]
        print(path)
        display(ds)
        # For the publication I actually saved these out locally (they are tiny) and uploaded them to OSN
        # Something got stuck in fsspec (I changed the aws creds mid computation) and I did not want to wait
        # for the computation on my small laptop gain
        # ds.to_zarr(f'./test_save/{path}', mode='w')
        # this should work with the proper credentials
        ds.to_zarr(fs.get_mapper(path))

scale-aware-air-sea/jbusecke/v1.0.1/plotting/CM26_histogram_qh.zarr


<xarray.Dataset>
Dimensions:             (algo: 5, Large Scale qh_bin: 499,
                         Small Scale qh_bin: 499, WBC_mask_bin: 2)
Coordinates:
  * algo                (algo) <U8 'ncar' 'ecmwf' ... 'coare3p6' 'andreas'
  * Large Scale qh_bin  (Large Scale qh_bin) float64 -1.198e+03 ... 295.5
  * Small Scale qh_bin  (Small Scale qh_bin) float64 -299.5 -298.5 ... 198.5
  * WBC_mask_bin        (WBC_mask_bin) float64 -0.25 1.25
Data variables:
    histogram_qh        (algo, Large Scale qh_bin, Small Scale qh_bin, WBC_mask_bin) int64 ...

scale-aware-air-sea/jbusecke/v1.0.1/plotting/CM26_histogram_ql.zarr


<xarray.Dataset>
Dimensions:             (algo: 5, Large Scale ql_bin: 499,
                         Small Scale ql_bin: 499, WBC_mask_bin: 2)
Coordinates:
  * algo                (algo) <U8 'ncar' 'ecmwf' ... 'coare3p6' 'andreas'
  * Large Scale ql_bin  (Large Scale ql_bin) float64 -1.198e+03 ... 295.5
  * Small Scale ql_bin  (Small Scale ql_bin) float64 -299.5 -298.5 ... 198.5
  * WBC_mask_bin        (WBC_mask_bin) float64 -0.25 1.25
Data variables:
    histogram_ql        (algo, Large Scale ql_bin, Small Scale ql_bin, WBC_mask_bin) int64 ...

scale-aware-air-sea/jbusecke/v1.0.1/plotting/CM26_histogram_q_total.zarr


...

<xarray.Dataset>
Dimensions:             (algo: 2, Large Scale qh_bin: 499,
                         Small Scale qh_bin: 499, WBC_mask_bin: 2)
Coordinates:
  * algo                (algo) <U5 'ncar' 'ecmwf'
  * Large Scale qh_bin  (Large Scale qh_bin) float64 -1.198e+03 ... 295.5
  * Small Scale qh_bin  (Small Scale qh_bin) float64 -299.5 -298.5 ... 198.5
  * WBC_mask_bin        (WBC_mask_bin) float64 -0.25 1.25
Data variables:
    histogram_qh        (algo, Large Scale qh_bin, Small Scale qh_bin, WBC_mask_bin) int64 ...

scale-aware-air-sea/jbusecke/v1.0.1/plotting/CESM_histogram_ql.zarr


<xarray.Dataset>
Dimensions:             (algo: 2, Large Scale ql_bin: 499,
                         Small Scale ql_bin: 499, WBC_mask_bin: 2)
Coordinates:
  * algo                (algo) <U5 'ncar' 'ecmwf'
  * Large Scale ql_bin  (Large Scale ql_bin) float64 -1.198e+03 ... 295.5
  * Small Scale ql_bin  (Small Scale ql_bin) float64 -299.5 -298.5 ... 198.5
  * WBC_mask_bin        (WBC_mask_bin) float64 -0.25 1.25
Data variables:
    histogram_ql        (algo, Large Scale ql_bin, Small Scale ql_bin, WBC_mask_bin) int64 ...

scale-aware-air-sea/jbusecke/v1.0.1/plotting/CESM_histogram_q_total.zarr


<xarray.Dataset>
Dimensions:                  (algo: 2, Large Scale q_total_bin: 499,
                              Small Scale q_total_bin: 499, WBC_mask_bin: 2)
Coordinates:
  * algo                     (algo) <U5 'ncar' 'ecmwf'
  * Large Scale q_total_bin  (Large Scale q_total_bin) float64 -1.198e+03 ......
  * Small Scale q_total_bin  (Small Scale q_total_bin) float64 -299.5 ... 198.5
  * WBC_mask_bin             (WBC_mask_bin) float64 -0.25 1.25
Data variables:
    histogram_q_total        (algo, Large Scale q_total_bin, Small Scale q_total_bin, WBC_mask_bin) int64 ...